# 12 Merge Related Words into One Lemma

People list the same idea in several forms (`gambling`, `gamble`; `luck`, `lucky`; `people`, `person`). The lemmatizer (step 05) leaves many of these apart. This step merges them into one **lemma** and keeps the original `feature` next to it, so both counts are visible (`feature | lemma | frequency_feature | frequency_lemma`), as in the 2019 norms.

The rules live in `merge_lemmas.py` (the LLM pipeline uses the same file), applied in this order:

1. **The 2019 translation** (`dictionaries/translation_2019.csv`), without the pairs that change the meaning: the word-to-root pairs used for the published 2019 norms (7,522 words; `luck`/`lucky` → `luck`, `personal`/`people` → `person`, `action` → `act`, `clothing`/`clothes` → `cloth`). Recovered from the `feature`/`translated` columns of the published file (`english_Buchanan2019.csv` in the subs2strudel history); each word maps to exactly one root. A pair is **not** merged when it joins opposites (`unhappy`/`happy`, `illegal`/`legal`), uses a function word as the root (`sometimes`/`some`, `without`/`with`), or the two words are neither related in WordNet nor a common word plus a transparent ending (`medicine`/`medic`, `death`/`dead`, `physical`/`physic`). 2,667 of the 7,522 pairs are dropped this way (`dictionaries/lemma_not_merged.csv`); sensible irregular pairs are always kept (`dictionaries/lemma_force_merge.csv`: `people`/`person`, `grey`/`gray`, `honesty`/`honest`).
2. **Words the 2019 data already saw and left alone** (`translation_2019_seen.txt`) and the keep-separate list (`dictionaries/lemma_keep_separate.csv`: `glasses`, `states`, ...) are not merged.
3. **Affix rules for every other word** (new words in these data, and words the language models write): the endings from the original 2019 affix list (`-ness`, `-ous`, `-ful`, `-ive`, `-ion`, `-ment`, `-er`, `-ly`, `-ing`, `-ed`, `-s`, ...). The base must be a known root (a 2019 root, or a word at least 5 people wrote, `dictionaries/lemma_roots.csv`) and WordNet must say the two are related. This blocks `united → unit`, `master → mast`, `corner → corn`. Tested against the 2019 dictionary itself, the rules reproduce 4,069 of its 7,522 merges, pick a different root for 22, and wrongly merge 22 of 3,276 words that 2019 left alone.

As in 2019, a feature that merges into the cue word itself is dropped for that cue (`people` for the cue `person`).

Outputs: `dictionaries/lemma_merge.csv` (every merge, with its source; the LLM pipeline reads the same file) and `processed/12_cue_feature_counts_merged.csv`.

| Column | Meaning |
|---|---|
| `cue` | the cue word |
| `feature` | the word as it came out of step 08 |
| `lemma` | the word it is merged into (the same word if nothing is merged) |
| `frequency_feature` | people who listed `feature` (same as step 09 `count`) |
| `frequency_lemma` | people who listed `lemma` or anything merged into it (a person who wrote two forms counts once) |
| `n` | people with at least one word for this cue (`n_final`) |
| `normalized_feature`, `normalized_lemma` | the two frequencies as a percent of `n` |
| `n_answered`, `pos`, `mean_position`, `original_words` | as in step 09 |

In [1]:
import pandas as pd
from merge_lemmas import Merger

In [2]:
words = pd.read_csv("processed/08_words.csv", dtype={"subject_id": str, "position": int,
                                                       "n_answered": int, "n_final": int}, keep_default_na=False)
n_word = words["word"].value_counts().to_dict()
print(f"{len(words):,} rows, {len(n_word):,} distinct words")

952,792 rows, 29,913 distinct words


In [3]:
# roots the affix rules may merge into: the 2019 roots plus every word at least 5 people wrote
t19 = pd.read_csv("dictionaries/translation_2019.csv", keep_default_na=False)
roots = {w: n for w, n in n_word.items() if n >= 5}
for w in set(t19["merge_to"]):
    roots[w] = max(roots.get(w, 0), 5)
pd.DataFrame({"word": list(roots), "count": list(roots.values())}).sort_values("word").to_csv("dictionaries/lemma_roots.csv", index=False)

merger = Merger("dictionaries")
rows = []
for w, n in n_word.items():
    m = merger.merge(w)
    if m != w:
        rows.append((w, m, n, n_word.get(m, 0), "translation_2019" if w in merger.table else "affix_rules"))
rows += [(w, m, 0, n_word.get(m, 0), "translation_2019") for w, m in merger.table.items() if w not in n_word]
merge = pd.DataFrame(rows, columns=["word", "merge_to", "n_word", "n_merge_to", "source"]).sort_values(["n_word", "word"], ascending=[False, True])
merge.to_csv("dictionaries/lemma_merge.csv", index=False)
# the 2019 pairs we deliberately do not merge, for review (move a row to lemma_force_merge.csv to merge it)
rej = pd.DataFrame(sorted(merger.rejected.items()), columns=["word", "2019_merge_to"])
rej["n_word"] = rej["word"].map(lambda w: n_word.get(w, 0))
rej.sort_values("n_word", ascending=False).to_csv("dictionaries/lemma_not_merged.csv", index=False)
print(f"{len(rej):,} of the 7,522 pairs in the 2019 translation are not merged (opposites, function words, meaning changes); see dictionaries/lemma_not_merged.csv")
in_data = merge[merge["n_word"] > 0]
print(f"{len(in_data):,} words in the data are merged, covering {in_data['n_word'].sum():,} rows ({in_data['n_word'].sum() / len(words):.1%})")
print(in_data.groupby("source")["n_word"].agg(["count", "sum"]))
merge.head(30)

2,667 of the 7,522 pairs in the 2019 translation are not merged (opposites, function words, meaning changes); see dictionaries/lemma_not_merged.csv
5,628 words in the data are merged, covering 127,048 rows (13.3%)
                  count     sum
source                         
affix_rules        2859   12410
translation_2019   2769  114638


,word,merge_to,n_word,n_merge_to,source
0,people,person,6650,5109,translation_2019
1,different,differ,2472,50,translation_2019
2,feeling,feel,1406,1684,translation_2019
3,action,act,1327,1259,translation_2019
4,building,build,1004,599,translation_2019
5,clothes,cloth,960,440,translation_2019
6,dangerous,danger,883,293,translation_2019
7,math,mathematics,785,116,translation_2019
8,information,inform,777,97,translation_2019
9,dirty,dirt,770,575,translation_2019


## Apply the merge and recount

In [4]:
mapping = dict(zip(merge["word"], merge["merge_to"]))
words = words[words["word"].map(lambda w: mapping.get(w, w)) != words["cue"]]   # a feature that merges into the cue itself is dropped
old = pd.read_csv("processed/09_cue_feature_counts.csv", keep_default_na=False)   # original counts
old = old[old["word"].map(lambda w: mapping.get(w, w)) != old["cue"]]

# people per (cue, merged word); a person who wrote both forms counts once
words["lemma"] = words["word"].map(lambda w: mapping.get(w, w))
merged = (words.groupby(["cue", "lemma"])["subject_id"].nunique()
               .rename("frequency_lemma").reset_index())

out = old.rename(columns={"word": "feature", "count": "frequency_feature", "n_final": "n"})
out["lemma"] = out["feature"].map(lambda w: mapping.get(w, w))
out = out.merge(merged, on=["cue", "lemma"], how="left")
out["normalized_feature"] = (out["frequency_feature"] / out["n"] * 100).round(4)
out["normalized_lemma"] = (out["frequency_lemma"] / out["n"] * 100).round(4)
out = out[["cue", "feature", "lemma", "frequency_feature", "frequency_lemma", "n",
           "normalized_feature", "normalized_lemma", "n_answered", "pos", "mean_position", "original_words"]]
out = out.sort_values(["cue", "frequency_lemma", "lemma", "frequency_feature"], ascending=[True, False, True, False])
assert (out["frequency_lemma"] >= out["frequency_feature"]).all()
assert (out["frequency_lemma"] <= out["n"]).all()
print(f"{len(out):,} cue-word rows; {(out['feature'] != out['lemma']).sum():,} are merged into another word")

428,979 cue-word rows; 65,859 are merged into another word


## Check: how much did the top features change?

In [5]:
merged_pairs = out.drop_duplicates(["cue", "lemma"])
def top10(df, col, word):
    return df.sort_values([col, word], ascending=[False, True]).groupby("cue").head(10)
o = top10(out, "frequency_feature", "feature")
m = top10(merged_pairs, "frequency_lemma", "lemma")
changed = set(zip(m["cue"], m["lemma"])) - set(zip(o["cue"], o["feature"]))
print(f"{len(changed):,} of {len(m):,} top-10 cue-word slots differ after merging")
out[out["cue"] == "chance"].head(12)

6,055 of 37,220 top-10 cue-word slots differ after merging


,cue,feature,lemma,frequency_feature,frequency_lemma,n,normalized_feature,normalized_lemma,n_answered,pos,mean_position,original_words
62882,chance,luck,luck,13,17,68,19.1176,25.0000,70,NOUN,2.46,luck
62897,chance,lucky,luck,4,17,68,5.8824,25.0000,70,ADJ,1.75,lucky
62881,chance,opportunity,opportune,17,17,68,25.0000,25.0000,70,NOUN,1.41,opportunity
62884,chance,gambling,gamble,8,12,68,11.7647,17.6471,70,NOUN,3.62,gambling
62893,chance,gamble,gamble,4,12,68,5.8824,17.6471,70,NOUN,2.25,gamble
62883,chance,risk,risk,12,12,68,17.6471,17.6471,70,NOUN,2.33,risk|risks
62885,chance,happen,happen,8,8,68,11.7647,11.7647,70,VERB,3.38,happen|happening|happens
62886,chance,probability,probable,8,8,68,11.7647,11.7647,70,NOUN,2.88,probability
62887,chance,game,game,7,7,68,10.2941,10.2941,70,NOUN,4.00,game|games
62888,chance,monopoly,monopoly,6,6,68,8.8235,8.8235,70,NOUN,3.67,monopoly


In [6]:
out.to_csv("processed/12_cue_feature_counts_merged.csv", index=False)
print(f"Saved {len(out):,} rows to processed/12_cue_feature_counts_merged.csv")

Saved 428,979 rows to processed/12_cue_feature_counts_merged.csv
